# 加减乘除初步体验

# add_abs 算子

## 概述

`add_abs` 是一个 PyPTO 逐元素（element-wise）动态算子，计算 $y = a + |b|$，即输入 `a` 与输入 `b` 的绝对值逐元素相加。

| 属性 | 值 |
|------|-----|
| 算子名称 | `add_abs` |
| 内核函数 | `add_abs_kernel` |
| 算子类型 | 逐元素（Element-wise） |
| 动态轴 | n 轴（支持运行时可变 n） |

---

## 数学公式

$$ y = a + |b| $$

对于任意位置 $(i, j)$：

$$ y[i][j] = a[i][j] + |\ b[i][j]\ | $$

---

## 张量规格

| 参数 | 形状 | 数据类型 | 说明 |
|------|------|----------|------|
| 输入 `a` | `[n, d]` | `float32` | 第一输入张量 |
| 输入 `b` | `[n, d]` | `float32` | 第二输入张量（取绝对值后与 a 相加） |
| 输出 `y` | `[n, d]` | `float32` | 计算结果张量 |

- **n 轴**: 动态轴，支持运行时指定任意正整数值。
- **d 轴**: 固定维度（编译时由输入 shape 决定）。

---

## 精度标准

| 指标 | 值 |
|------|-----|
| 绝对容差 (atol) | `0.000025` |
| 相对容差 (rtol) | `0.005` |

---

## 实现细节

### 内核签名

```python
@pypto.frontend.jit(runtime_options={"run_mode": global_run_mode})
def add_abs_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)
):
```

- **JIT 编译**: 使用 `pypto.frontend.jit` 装饰器自动编译。
- **动态形状**: 张量声明使用 `Tensor([], ...)` 表示动态 shape 支持。
- **Tile 配置**: `set_vec_tile_shapes(2, 8)` 设置向量 tile 形状。

### 核心计算

```python
# torch
  # y = a + torch.abs(b)

# pypto: 
  # tiling
  # operator
```

**涉及的 PyPTO API**：
| API | 用途 | 约束 |
|-----|------|------|
| `pypto.set_vec_tile_shapes` | 设置基本块tiling | 尾轴必须32B对齐 |
| `pypto.abs` | 计算张量 `b` 的逐元素绝对值 | |
| `pypto.add` | 将 `a` 与 `abs(b)` 逐元素相加 | |

**API 使用指导**：
- [PyPTO TileShape 设置方法](https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_vec_tile_shapes.md)
- [PyPTO 绝对值 abs 接口](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-abs.md)
- [PyPTO 加法 add 接口](https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-add.md)

## 测试用例

| 测试 ID | 名称 | 说明 |
|---------|------|------|
| `add_abs::test_add_abs_basic` | 基础功能验证 | 固定小张量 `[2, 2]` 验证 $y = a + \|b\|$ 的正确性 |
| `add_abs::test_add_abs_dynamic_n` | 动态 n 轴验证 | 测试 `n ∈ {3, 7, 15}` 运行时动态 shape，验证算子对不同 n 值的正确性 |
| `add_abs::test_add_abs_edge_cases` | 边界场景验证 | 覆盖全零输入、b 全部为负数、大数值等边界情况 |

---

## 注意

1. **Tile 配置**: 当前使用固定 `set_vec_tile_shapes(2, 8)`，尾轴需要32B对齐。

In [24]:
import pypto
import torch
import numpy as np
from numpy.testing import assert_allclose


# ============================================================================
# Kernel Definition
# ============================================================================

@pypto.frontend.jit
def add_abs_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    """y = a + |b|"""
    # TODO: 请在下方添加 PyPTO 实现代码
    # 设置 tiling(即 set_vec_tile_shapes)
    ...

    # 实现 out[:] = a + |b|
    ...


# ============================================================================
# Test Cases
# ============================================================================

def test_add_abs_basic():
    """基础测试: y = a + |b|"""
    a = torch.tensor([[1.0, -2.0], [3.0, -4.0]], dtype=torch.float32, device='npu:0')
    b = torch.tensor([[-2.0, 3.0], [-4.0, 5.0]], dtype=torch.float32, device='npu:0')
    expected = a + torch.abs(b)

    out = torch.empty(a.shape, dtype=torch.float32, device='npu:0')
    add_abs_kernel(a, b, out)
    assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print(f"Input a:  {a}")
    print(f"Input b:  {b}")
    print(f"Output:   {out}")
    print(f"Expected: {expected}")
    print("✓ Basic test passed")


def test_add_abs_dynamic_n():
    """动态 n-axis 测试"""
    for n in [3, 7, 15]:
        a = torch.randn(n, 4, dtype=torch.float32, device='npu:0')
        b = torch.randn(n, 4, dtype=torch.float32, device='npu:0')
        expected = a + torch.abs(b)

        out = torch.empty(a.shape, dtype=torch.float32, device='npu:0')
        add_abs_kernel(a, b, out)
        assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)

        max_diff = np.abs(out.cpu().numpy() - expected.cpu().numpy()).max()
        print(f"  n={n:3d}: max_diff={max_diff:.8f} ✓")


def test_add_abs_edge_cases():
    """边界测试"""
    # 1. 全零
    a = torch.zeros((2, 3), dtype=torch.float32, device='npu:0')
    b = torch.zeros((2, 3), dtype=torch.float32, device='npu:0')
    expected = a + torch.abs(b)
    out = torch.empty(a.shape, dtype=torch.float32, device='npu:0')
    add_abs_kernel(a, b, out)
    assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print("  [zeros] ✓")

    # 2. b 全负
    a = torch.ones((2, 3), dtype=torch.float32, device='npu:0')
    b = torch.full((2, 3), -3.0, dtype=torch.float32, device='npu:0')
    expected = a + torch.abs(b)
    out = torch.empty(a.shape, dtype=torch.float32, device='npu:0')
    add_abs_kernel(a, b, out)
    assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print("  [neg_b] ✓")

    # 3. 大值
    a = torch.tensor([[100.0, -200.0]], dtype=torch.float32, device='npu:0')
    b = torch.tensor([[-50.0, 150.0]], dtype=torch.float32, device='npu:0')
    expected = a + torch.abs(b)
    out = torch.empty(a.shape, dtype=torch.float32, device='npu:0')
    add_abs_kernel(a, b, out)
    assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print("  [large] ✓")


# ============================================================================
# Main
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("PyPTO add_abs Operator Tests (Simplified)")
    print("=" * 60)

    test_add_abs_basic()
    test_add_abs_dynamic_n()
    test_add_abs_edge_cases()

    print("=" * 60)
    print("All tests passed!")
    print("=" * 60)

PyPTO add_abs Operator Tests (Simplified)
Input a:  tensor([[ 1., -2.],
        [ 3., -4.]], device='npu:0')
Input b:  tensor([[-2.,  3.],
        [-4.,  5.]], device='npu:0')
Output:   tensor([[3., 1.],
        [7., 1.]], device='npu:0')
Expected: tensor([[3., 1.],
        [7., 1.]], device='npu:0')
✓ Basic test passed
  n=  3: max_diff=0.00000000 ✓
  n=  7: max_diff=0.00000000 ✓
  n= 15: max_diff=0.00000000 ✓
  [zeros] ✓
  [neg_b] ✓
  [large] ✓
All tests passed!


## 用户需求(For AI Agent)
本文档记录了用户开发算子的需求，请根据描述进行需求检测及分析。

---

## 需求-1
请开发一个 PyPTO 动态算子：
- 算子名称：add_abs
- 公式：$ y = a + |b| $
- 规格：

| 类型  | shape  | dtype  |
| ------------ | ------------ | ------------ |
| 输入 a| \[n, d\]  | float32  |
| 输入 b| \[n, d\]  | float32  |
| 输出 y| \[n, d\]  | float32  |

- 精度标准：atol=0.000025, rtol=0.005
- 动态轴：n轴

---